# Merkle DE Case Study

This notebook implements the three-layer data lake and `top_item` datamart on Databricks Free Edition.

| Layer | Schema | Output |
|---|---|---|
| 1 – Raw | `layer1_raw` | `item_raw`, `event_raw`: source data as-is, all columns `STRING` |
| 2 – Curated | `layer2_curated` | `item_curated`, `event_curated`: typed and flattened |
| 3 – Mart | `layer3_mart` | `top_item` datamart |

All schemas live in the dedicated catalog `merkle_de`. The source data profiling, the supporting design decisions and
post-run checks of the results are in `data_exploration.ipynb`.

## 0. Configuration

All names used in the notebook are defined here: the catalog, the source path and the five tables.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

CATALOG = "merkle_de"
SOURCE_PATH = "s3://merkle-de-interview-case-study/de"

L1_ITEM = f"{CATALOG}.layer1_raw.item_raw"
L1_EVENT = f"{CATALOG}.layer1_raw.event_raw"
L2_ITEM = f"{CATALOG}.layer2_curated.item_curated"
L2_EVENT = f"{CATALOG}.layer2_curated.event_curated"
TOP_ITEM = f"{CATALOG}.layer3_mart.top_item"

## 1. Catalog and schemas

The project has its own catalog, with one schema per layer.

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG} COMMENT 'Merkle DE case study'")
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.layer1_raw "
    "COMMENT 'Layer 1: raw source data, all STRING, no transformations'"
)
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.layer2_curated "
    "COMMENT 'Layer 2: curated data - renamed, flattened, typed, partitioned'"
)
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.layer3_mart COMMENT 'Layer 3: datamarts'"
)

## 2. Layer 1 – raw tables

Spark reads both files directly from the public S3 bucket. In production the bucket would be registered as a Unity
Catalog external location, and new files would be loaded incrementally with Auto Loader.

Column names match the file headers exactly, including the dotted `event.payload`, and every column is `STRING`.

Two reader options matter:

* **No schema inference.** Spark then reads every column as `STRING`. Empty fields (e.g. a missing `adjective`)
  are stored as `NULL`.
* **`escape = '"'`.** The JSON in `event.payload` uses CSV-style doubled quotes (`""`). Setting `escape='"'` ensures that the quoted JSON field is parsed correctly.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {L1_ITEM} (
    adjective   STRING,
    category    STRING,
    created_at  STRING,
    id          STRING,
    modifier    STRING,
    name        STRING,
    price       STRING
)
COMMENT 'Layer 1: raw copy of item.csv'
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {L1_EVENT} (
    event_id         STRING,
    event_time       STRING,
    user_id          STRING,
    `event.payload`  STRING
)
COMMENT 'Layer 1: raw copy of event.csv (one row per event parameter)'
""")

In [ ]:
# Read both files from S3 without schema inference (every column STRING) and load them into layer 1
raw_item = spark.read.csv(f"{SOURCE_PATH}/item.csv", header=True, escape='"')
raw_event = spark.read.csv(f"{SOURCE_PATH}/event.csv", header=True, escape='"')

# insertInto matches columns by position, so the columns are selected by name in DDL order:
# a renamed or missing source column fails here instead of shifting values between columns
raw_item.select(
    "adjective",
    "category",
    "created_at",
    "id",
    "modifier",
    "name",
    "price",
).write.insertInto(L1_ITEM, overwrite=True)

raw_event.select(
    "event_id",
    "event_time",
    "user_id",
    "`event.payload`",
).write.insertInto(L1_EVENT, overwrite=True)

## 3. Layer 2 – curated tables

**Naming convention:** all names are `snake_case`, and identifiers are called `<entity>_id` (`id` → `item_id`).
Descriptive attributes keep their source names. Table names carry the layer as a suffix (`item_raw` →
`item_curated`).

**Data types**

* **IDs** become `BIGINT`. User and event IDs in tracking data grow quickly, and widening a column type later
  means rewriting the Delta table.
  * `item.id` and `user_id` are stored as decimal-formatted strings such as `"2512.0"`. With ANSI mode on (the
    serverless default), a direct `CAST('2512.0' AS BIGINT)` fails, so these IDs are cast to `DOUBLE` first and
    then to `BIGINT`.
  * The `DOUBLE` step would silently truncate a value like `2512.5`. The validation in section 5 therefore checks
    that no source ID has a fractional part.
  * IDs inside the payload are plain digits and are cast directly.
* **`price`** becomes `DECIMAL(10,2)`. The source carries floating-point noise (e.g. `213.60000000000002`), and the
  interviewer confirmed that their systems operate on two decimal places.
* **`created_at` and `event_time`** become `TIMESTAMP_NTZ`. The source contains no time-zone information, so `TIMESTAMP_NTZ` preserves the timestamps without introducing time-zone semantics.
* **`test_assignment`** remains `INT` to avoid assuming Boolean semantics that are not defined in the source.

**Partitioning:** the fact table `event_curated` is partitioned by `event_year`.
* The datamart aggregates by year.
* Daily partitions would mean about 1,900 tiny partitions for this small dataset.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {L2_ITEM} (
    item_id     BIGINT         NOT NULL COMMENT 'Item identifier (source: id)',
    item_name   STRING                  COMMENT 'adjective + category + modifier; not unique (source: name)',
    adjective   STRING                  COMMENT 'e.g. fuzzy; may be NULL',
    category    STRING                  COMMENT 'e.g. gadget',
    modifier    STRING                  COMMENT 'e.g. charger; may be NULL',
    price       DECIMAL(10,2)           COMMENT 'Item price',
    created_at  TIMESTAMP_NTZ           COMMENT 'Item creation timestamp; source contains no time-zone information'
)
COMMENT 'Layer 2: item dimension, one row per item'
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {L2_EVENT} (
    event_id         STRING         NOT NULL COMMENT 'Event identifier',
    event_time       TIMESTAMP_NTZ  NOT NULL COMMENT 'Event time as provided (no time zone)',
    event_year       INT            NOT NULL COMMENT 'Year of event_time; partition column',
    user_id          BIGINT                  COMMENT 'User who triggered the event',
    event_name       STRING                  COMMENT 'view_item | view_user_profile | test_assignment',
    platform         STRING                  COMMENT 'web | android | iOS | mobile web | server',
    item_id          BIGINT                  COMMENT 'Viewed item (view_item only)',
    referrer         STRING                  COMMENT 'Where the item view came from (view_item only)',
    test_id          BIGINT                  COMMENT 'Test identifier (test_assignment events only)',
    test_assignment  INT                     COMMENT 'Test assignment value (observed: 0/1; test_assignment events only)',
    viewed_user_id   BIGINT                  COMMENT 'Viewed user profile (view_user_profile only)'
)
PARTITIONED BY (event_year)
COMMENT 'Layer 2: event fact, one row per event'
""")

### 3.1 `item_curated`

`item` has no nested attributes, so this step only renames and casts.

In [ ]:
item = spark.table(L1_ITEM).select(
    F.col("id").cast("double").cast("bigint").alias("item_id"),  # "2512.0" -> 2512
    F.col("name").alias("item_name"),
    "adjective",
    "category",
    "modifier",
    F.col("price").cast("decimal(10,2)").alias("price"),  # rounds to 2 decimals
    F.col("created_at").cast("timestamp_ntz").alias("created_at"),
)

# insertInto matches columns by position; the select above follows the DDL column order
item.write.insertInto(L2_ITEM, overwrite=True)

### 3.2 `event_curated`: flatten the payload and pivot

`event.payload` is a JSON object with four keys: `event_name`, `platform`, `parameter_name` and `parameter_value`.
Each event type has a fixed set of parameters:

| event_name | parameters (one source row each) |
|---|---|
| `view_item` | `item_id`, `referrer` |
| `test_assignment` | `test_id`, `test_assignment` |
| `view_user_profile` | `viewed_user_id` |

The transformation has three steps:

1. Parse the JSON into columns.
2. Pivot the parameter rows into **one row per event**, so each parameter gets its own column (`NULL` where it doesn't apply).
3. Cast the columns to their types.

Within an event, the event attributes (time, user, name, platform) are identical across rows, and each parameter
appears only once. `first()` therefore loses nothing, which the validation in section 5 confirms.

In [ ]:
payload_schema = (
    "event_name STRING, platform STRING, parameter_name STRING, parameter_value STRING"
)
parameters = ["item_id", "referrer", "test_id", "test_assignment", "viewed_user_id"]

event = (
    spark.table(L1_EVENT)
    .withColumn("payload", F.from_json(F.col("`event.payload`"), payload_schema))
    .select("event_id", "event_time", "user_id", "payload.*")
    .groupBy("event_id", "event_time", "user_id", "event_name", "platform")
    .pivot("parameter_name", parameters)
    .agg(F.first("parameter_value"))
    .withColumn("event_time", F.col("event_time").cast("timestamp_ntz"))
    .select(
        "event_id",
        "event_time",
        F.year("event_time").alias("event_year"),
        # "178481.0" -> 178481
        F.col("user_id").cast("double").cast("bigint").alias("user_id"),
        "event_name",
        "platform",
        F.col("item_id").cast("bigint").alias("item_id"),
        "referrer",
        F.col("test_id").cast("bigint").alias("test_id"),
        F.col("test_assignment").cast("int").alias("test_assignment"),
        F.col("viewed_user_id").cast("bigint").alias("viewed_user_id"),
    )
)

# insertInto matches columns by position; the select above follows the DDL column order
event.write.insertInto(L2_EVENT, overwrite=True)

## 4. Layer 3 – `top_item` datamart

`top_item` contains yearly view metrics for each item with at least one view.

**Grain:** one row per `item_id` and `view_year`.

| Attribute | Type | Description |
|---|---|---|
| `item_id` | `BIGINT` | Item identifier |
| `view_year` | `INT` | Calendar year derived from `event_time` |
| `total_views` | `BIGINT` | Number of `view_item` events for the item in that year |
| `view_rank` | `INT` | Rank by yearly views; ties share the same rank (`RANK()`) |
| `most_used_platforms` | `ARRAY<STRING>` | Platform(s) with the most views; all ties are included and sorted alphabetically |

### Business rules

- Only item/year combinations with at least one view are included; combinations with zero views are omitted.
  **Note for the visualisation team:** a missing item/year combination means zero views.
- One `view_item` event represents one view.
- Items with equal yearly view counts share the same rank.
- If several platforms have the same maximum number of views, all are included in `most_used_platforms`.

### Data notes

- 2013 and 2018 contain partial-year data: events run from 2013-03-09 to 2018-06-01.
- 3,148 views occur before the corresponding item's `created_at`; they are preserved rather than filtered.

### DDL

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {TOP_ITEM} (
    item_id              BIGINT         NOT NULL COMMENT 'Item identifier, joins to layer2_curated.item_curated',
    view_year            INT            NOT NULL COMMENT 'Year of the views (from event_time)',
    total_views          BIGINT         NOT NULL COMMENT 'Number of item views in the year',
    view_rank            INT            NOT NULL COMMENT 'RANK() by total_views within the year; ties share a rank',
    most_used_platforms  ARRAY<STRING>  NOT NULL COMMENT 'Platform(s) with most views in the year; all ties, sorted'
)
COMMENT 'Layer 3: yearly item views, rank and most used platform(s); only item/year combinations with views'
""")

### Load

The load has three steps:

1. Count the yearly views per item.
2. Rank the platforms by their view count for each item and year, and collect the top platform(s) into a sorted
   array. Tied platforms are all kept.
3. Rank the items within each year by their views.

In [ ]:
views = (
    spark.table(L2_EVENT)
    .filter(F.col("event_name") == "view_item")
    .select("item_id", F.col("event_year").alias("view_year"), "platform")
)

# 1. Yearly views per item
item_year = views.groupBy("item_id", "view_year").agg(F.count("*").alias("total_views"))

# 2. Platform(s) with the most views per item and year
platform_views = views.groupBy("item_id", "view_year", "platform").agg(
    F.count("*").alias("platform_views")
)
platform_window = Window.partitionBy("item_id", "view_year").orderBy(
    F.desc("platform_views")
)
top_platforms = (
    platform_views.withColumn("platform_rank", F.rank().over(platform_window))
    .filter(F.col("platform_rank") == 1)
    .groupBy("item_id", "view_year")
    .agg(F.array_sort(F.collect_list("platform")).alias("most_used_platforms"))
)

# 3. Rank items within each year; items with equal views share a rank
year_window = Window.partitionBy("view_year").orderBy(F.desc("total_views"))
top_item = item_year.join(top_platforms, ["item_id", "view_year"]).select(
    "item_id",
    "view_year",
    "total_views",
    F.rank().over(year_window).alias("view_rank"),
    "most_used_platforms",
)

# insertInto matches columns by position; the select above follows the DDL column order
top_item.write.insertInto(TOP_ITEM, overwrite=True)

## 5. Validation

These checks cover what would make the datamart wrong:

* **No ID truncated.** Source IDs written as decimals (`"2512.0"`) must be whole numbers, so the `DOUBLE` → `BIGINT`
  cast drops nothing.
* **No events lost or duplicated.** Layer 2 must have exactly one row per source `event_id`.
* **The pivot kept every parameter.** The parameter values in layer 2 must match the layer 1 row count one for one.
  This also catches an unknown parameter name or a repeated parameter.
* **Views reconcile.** Every view must point to an existing item, and `top_item` must add up to all view events,
  with one row per item and year.

In [ ]:
l1_item = (
    spark.table(L1_ITEM)
    .agg(
        F.count_if(F.col("id").cast("double") % 1 != 0).alias("fractional_ids"),
    )
    .first()
)

l1_event = (
    spark.table(L1_EVENT)
    .agg(
        F.count_distinct("event_id").alias("events"),
        F.count("*").alias("parameter_rows"),
        F.count_if(F.col("user_id").cast("double") % 1 != 0).alias(
            "fractional_user_ids"
        ),
    )
    .first()
)

# Each raw parameter row must become exactly one populated parameter column after the pivot,
# so the non-NULL parameter values in layer 2 must equal the layer 1 row count
# (source parameter values are non-NULL).
l2_event = (
    spark.table(L2_EVENT)
    .agg(
        F.count("*").alias("events"),
        F.count_distinct("event_id").alias("distinct_events"),
        (
            F.count("item_id")
            + F.count("referrer")
            + F.count("test_id")
            + F.count("test_assignment")
            + F.count("viewed_user_id")
        ).alias("parameter_values"),
        F.count_if(F.col("event_name") == "view_item").alias("views"),
    )
    .first()
)

views_without_item = (
    spark.table(L2_EVENT)
    .filter(F.col("event_name") == "view_item")
    .join(spark.table(L2_ITEM), "item_id", "left_anti")
    .count()
)

top_item_check = (
    spark.table(TOP_ITEM)
    .agg(
        F.sum("total_views").alias("views"),
        F.count("*").alias("rows"),
        F.count_distinct("item_id", "view_year").alias("item_years"),
    )
    .first()
)

print("layer 1 item: ", l1_item.asDict())
print("layer 1 event:", l1_event.asDict())
print("layer 2 event:", l2_event.asDict())
print("views without item:", views_without_item)
print("top_item:     ", top_item_check.asDict())

assert l1_item.fractional_ids == 0, "item ids with a fractional part"
assert l1_event.fractional_user_ids == 0, "user ids with a fractional part"
assert (
    l2_event.events == l2_event.distinct_events == l1_event.events
), "events lost or duplicated"
assert (
    l2_event.parameter_values == l1_event.parameter_rows
), "parameters lost in the pivot"
assert views_without_item == 0, "views of unknown items"
assert top_item_check.views == l2_event.views, "top_item does not add up to all views"
assert (
    top_item_check.rows == top_item_check.item_years
), "top_item has duplicate item/year rows"
print("All checks passed")